# V16 — DINOv3 ViT-7B final-block Q/V LoRA, fold-0 screen

**Status: siap dijalankan di Kaggle, belum dieksekusi.** Notebook ini tidak memuat atau membuat prediksi test. Semua angka hasil run harus berasal dari Kaggle T4×2.

V15 menutup jalur head/readout: 248 error salah di semua eksperimen, 212 error salah di kelima seed, dan pada 287 error champion tetangga terdekat lebih sering membawa kelas prediksi yang salah (183) daripada kelas benar (98). Jadi eksperimen ini mengubah representasi secara terbatas, bukan menambah readout baru.

## Hipotesis dan satu-satunya perubahan

Resep MIDOG DINOv3 memakai LoRA Q/V `r=8`, `alpha=16`, dropout `0.05`. ViT-7B penuh tidak realistis di T4×2, sehingga screen ini memasang resep itu **hanya pada fused QKV blok transformer terakhir (blok 39)**. Semua parameter pretrained asli tetap `requires_grad=False`; yang dilatih hanya empat matriks low-rank baru. Head V10 fold 0 dibangun ulang dari cache terverifikasi, harus mereproduksi kontrol, lalu dibekukan.

| Item | Nilai terkunci sebelum hasil |
|---|---|
| Backbone | timm `vit_7b_patch16_dinov3.lvd1689m`, revision V10 `b40f6c82…`, FP16 |
| Preprocess/readout | V10 persis: direct-square 256 bilinear; final CLS ‖ mean 256 true-patch token |
| Fold/label | fold 0, `StratifiedKFold(5, shuffle=True, random_state=42)`, 100% label resmi mentah |
| Adapter | blok 39 Q dan V saja, `r=8`, `alpha=16`, dropout `0.05` |
| Trainable | 131.072 parameter adapter; seluruh bobot pretrained dan head V10 beku |
| Optimisasi | 1 epoch, AdamW `1e-4`, wd `0.1`, warmup 10% + cosine, effective batch 16, clip 1.0 |
| Gate fold 0 | gain ≥ +0.0010; total error turun; R/O turun; Electronic F1 turun ≤ 0.0005 |

## Mengapa muat

Input dan seluruh blok 0–38 tidak membutuhkan gradien, sehingga autograd tidak menyimpan graph untuk bagian itu. Graph baru dimulai pada cabang LoRA di blok 39; tidak ada gradient checkpointing atau recompute 40 blok. Adapter FP32 + grad + Adam state hanya beberapa MiB. Berdasarkan throughput ekstraksi V13, dua validasi fold 0, satu epoch train, download/load, dan backward blok terakhir diperkirakan sekitar **1,8–2,3 jam**, bukan estimasi lama 4,4 jam per epoch yang mengasumsikan backward seluruh backbone.

## Sebelum Run All

Pasang accelerator **GPU T4×2**, Internet ON, Kaggle Secret `HF_TOKEN` dengan lisensi DINOv3 sudah diterima, dataset resmi, dan dataset cache `timm-embedding-satdat` yang dipakai V15. Notebook fail-closed bila cache tidak mereproduksi V10, revision berubah, sharding bukan dua rentang GPU kontigu, ada bobot pretrained yang trainable, atau kontrol live zero-LoRA tidak menghasilkan fold-0 V10: **0.9897706675032462, 62 error, 55 R/O**.

Jangan submit apa pun dari screen ini. Salin blok `V16 RESULT BLOCK` setelah run dan kirim balik; notebook berikutnya baru boleh dibuat setelah hasil itu diterima.

In [ ]:
%pip install -q "timm==1.0.26" "accelerate==1.13.0" "safetensors==0.7.0" "huggingface_hub>=0.34,<2"

In [ ]:
import os

os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import gc
import hashlib
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image, ImageFile, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2
from torchvision.transforms.functional import InterpolationMode
from tqdm.auto import tqdm

EXPERIMENT_NAME = 'v16_dinov3_vit7b_final_block_qv_lora_fold0'
REVISION_MARKER = 'v16_block39_qv_r8_a16_one_epoch_v1'
TIMM_MODEL_NAME = 'vit_7b_patch16_dinov3.lvd1689m'
TIMM_HF_REPO_ID = 'timm/vit_7b_patch16_dinov3.lvd1689m'
V10_MODEL_REVISION = 'b40f6c827cb02b13c7a23a76cf598489bb0f94e5'

IMAGE_SIZE = 256
HIDDEN_SIZE = 4096
PREFIX_TOKENS = 5
PATCH_TOKENS = 256
SEQUENCE_TOKENS = PREFIX_TOKENS + PATCH_TOKENS
FEATURE_DIM = 2 * HIDDEN_SIZE
BACKBONE_DTYPE = torch.float16
FEATURE_DTYPE = torch.float16
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

CLASS_DIR_TO_LABEL = {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}
CLASS_NAMES = ('Recyclable', 'Electronic', 'Organic')
NUM_CLASSES = 3
EXPECTED_COUNTS = {0: 9999, 1: 3961, 2: 12567}
EXPECTED_ROWS = 26527
ALLOWED_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
FOLD_SEED = 42
N_FOLDS = 5
DEVELOPMENT_FOLD = 0

EXPECTED_ORDER_SHA256 = '210f8f14797be49bad962e81f1da660a257ec28ee2136b54a7516950a66cb2e8'
EXPECTED_LABELS_SHA256 = '719d36e9918b1fd09711c06d1b259fca08ca6eb4d973de1388d5e3ddecf22d35'
EXPECTED_RAW_SHA256 = '73bd0d3c17e75e7c0f10e4324ffa9ef81f6224cd2b3ba0e53e9f2aea7fde3212'
EXPECTED_FOLD_SHA256 = '0a06976e5dc3cdb0b3f1f5043ff033bf95a40dd2b0cb82154bbf34d701ee305c'
V10_FOLD0 = {
    'macro_f1': 0.9897706675032462,
    'total_errors': 62,
    'ro_errors': 55,
    'f1_electronic': 0.9955891619407687,
}

HEAD_HIDDEN = 512
HEAD_DROPOUT = 0.40
HEAD_MAX_EPOCHS = 15
HEAD_PATIENCE = 5
HEAD_LR = 8e-4
HEAD_WEIGHT_DECAY = 1e-4
HEAD_BATCH = 256
HEAD_PREDICT_BATCH = 512
LABEL_SMOOTHING = 0.05

LORA_BLOCK_INDEX = 39
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_EPOCHS = 1
LORA_LR = 1e-4
LORA_WEIGHT_DECAY = 0.1
LORA_MICROBATCH = 4
LORA_GRAD_ACCUM = 4
LORA_EFFECTIVE_BATCH = LORA_MICROBATCH * LORA_GRAD_ACCUM
LORA_WARMUP_FRACTION = 0.10
GRAD_CLIP = 1.0
VALID_BATCH = 4
LOADER_WORKERS = 2

ADOPTION_MIN_GAIN = 0.0010
ADOPTION_MAX_ELECTRONIC_F1_DROP = 0.0005
RELOAD_TOLERANCE = 1e-6
EXPECTED_ADAPTER_PARAMETERS = 4 * HIDDEN_SIZE * LORA_RANK

LARGE_IMAGE_MP_THRESHOLD = 8_000_000
LARGE_IMAGE_MAX_SIDE = 4096
SAFE_DECODE_MAX_SIDE = 1024
Image.MAX_IMAGE_PIXELS = 200_000_000
ImageFile.LOAD_TRUNCATED_IMAGES = False

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
if not KAGGLE_INPUT.is_dir() or not KAGGLE_WORKING.is_dir():
    raise RuntimeError('This notebook must run on Kaggle with the official dataset attached.')
if not torch.cuda.is_available() or torch.cuda.device_count() != 2:
    raise RuntimeError('Set the Kaggle accelerator to GPU T4 x2 (exactly two CUDA devices).')
for device_id in (0, 1):
    name = torch.cuda.get_device_name(device_id)
    if 'T4' not in name.upper():
        raise RuntimeError('cuda:{} is {}, not a Tesla T4.'.format(device_id, name))

ARTIFACT_DIR = KAGGLE_WORKING / EXPERIMENT_NAME
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_STARTED = time.perf_counter()
TEST_ACCESS_STATE = {'test_images_accessed': False, 'test_metadata_accessed': False}


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(FOLD_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_float32_matmul_precision('high')


def json_default(value):
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError('Not JSON serializable: {}'.format(type(value).__name__))


def sha256_json(value):
    encoded = json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False,
                         default=json_default).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def write_json_atomic(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    with temporary.open('w', encoding='utf-8') as handle:
        json.dump(value, handle, indent=2, sort_keys=True, default=json_default, allow_nan=False)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def write_csv_atomic(frame, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


def write_npy_atomic(array, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp.npy')
    np.save(temporary, array)
    os.replace(temporary, path)


def atomic_torch_save(value, path):
    path = Path(path)
    temporary = path.with_name(path.name + '.tmp')
    if temporary.exists():
        temporary.unlink()
    with temporary.open('wb') as handle:
        torch.save(value, handle)
        handle.flush()
        os.fsync(handle.fileno())
    os.replace(temporary, path)


def read_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)


def walk_directories_without_test(base, max_depth=8):
    found, stack = [], [(Path(base), 0)]
    while stack:
        current, depth = stack.pop()
        try:
            children = list(current.iterdir())
        except OSError:
            continue
        for child in children:
            if not child.is_dir() or child.name.lower() == 'test':
                continue
            found.append(child)
            if depth < max_depth:
                stack.append((child, depth + 1))
    return found


def exact_raw_train_signature(directory):
    try:
        root_entries = list(directory.iterdir())
        if {p.name for p in root_entries} != set(CLASS_DIR_TO_LABEL):
            return None
        counts = {}
        for folder, label in CLASS_DIR_TO_LABEL.items():
            entries = list((directory / folder).iterdir())
            if any(p.is_symlink() or not p.is_file() or p.suffix.lower() not in ALLOWED_SUFFIXES
                   for p in entries):
                return None
            counts[label] = len(entries)
        return counts if counts == EXPECTED_COUNTS else None
    except OSError:
        return None


def resolve_raw_train():
    candidates = [p for p in walk_directories_without_test(KAGGLE_INPUT)
                  if p.name.lower() == 'train' and exact_raw_train_signature(p) == EXPECTED_COUNTS]
    unique = sorted({p.resolve() for p in candidates}, key=str)
    if len(unique) != 1:
        raise RuntimeError('Expected exactly one official raw train directory; found: {}'.format(unique))
    return unique[0]


def index_raw_train(train_dir):
    rows = []
    for folder, label in CLASS_DIR_TO_LABEL.items():
        for path in sorted((train_dir / folder).iterdir(), key=lambda p: p.name):
            rows.append({'image_id': path.relative_to(train_dir).as_posix(), 'label': label,
                         'path': str(path.resolve()), 'file_size_bytes': path.stat().st_size})
    frame = pd.DataFrame(rows).sort_values('image_id').reset_index(drop=True)
    if len(frame) != EXPECTED_ROWS or not frame['image_id'].is_unique:
        raise RuntimeError('Official train row count or image_id uniqueness failed.')
    return frame


TRAIN_DIR = resolve_raw_train()
train_df = index_raw_train(TRAIN_DIR)
train_df['fold'] = -1
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=FOLD_SEED)
for fold, (_, valid_positions) in enumerate(splitter.split(train_df, train_df['label'])):
    train_df.loc[valid_positions, 'fold'] = fold
train_df['fold'] = train_df['fold'].astype(np.int64)

ORDER_SHA256 = sha256_json(train_df['image_id'].tolist())
LABELS_SHA256 = sha256_json(train_df['label'].astype(int).tolist())
RAW_SHA256 = sha256_json(train_df[['image_id', 'label', 'file_size_bytes']].to_dict('records'))
FOLD_SHA256 = sha256_json(train_df[['image_id', 'label', 'fold']].to_dict('records'))
if (ORDER_SHA256, LABELS_SHA256, RAW_SHA256, FOLD_SHA256) != (
        EXPECTED_ORDER_SHA256, EXPECTED_LABELS_SHA256, EXPECTED_RAW_SHA256, EXPECTED_FOLD_SHA256):
    raise RuntimeError('Official data/fold contract differs from V10; refusing to continue.')

LABEL_VALUES = train_df['label'].to_numpy(dtype=np.int64)
FOLD_VALUES = train_df['fold'].to_numpy(dtype=np.int64)
TRAIN_INDICES = np.flatnonzero(FOLD_VALUES != DEVELOPMENT_FOLD)
VALID_INDICES = np.flatnonzero(FOLD_VALUES == DEVELOPMENT_FOLD)
write_csv_atomic(train_df[['image_id', 'label', 'fold']], ARTIFACT_DIR / 'fold_assignments.csv')

IMAGE_TRANSFORM = v2.Compose([
    v2.ToImage(),
    v2.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def load_rgb_image(path):
    try:
        with Image.open(path) as image:
            width, height = image.size
            if width <= 0 or height <= 0:
                raise ValueError('invalid dimensions')
            needs_guard = (width * height >= LARGE_IMAGE_MP_THRESHOLD
                           or max(width, height) > LARGE_IMAGE_MAX_SIDE)
            image.draft('RGB', (SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE))
            rgb = ImageOps.exif_transpose(image).convert('RGB')
            if needs_guard:
                rgb.thumbnail((SAFE_DECODE_MAX_SIDE, SAFE_DECODE_MAX_SIDE), Image.Resampling.LANCZOS)
            return rgb.copy()
    except Exception as exc:
        raise RuntimeError('Cannot decode {}'.format(path)) from exc


class RawImageRows(Dataset):
    def __init__(self, frame, indices):
        self.frame = frame
        self.indices = [int(i) for i in indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        row = self.frame.iloc[index]
        return IMAGE_TRANSFORM(load_rgb_image(row['path'])), int(row['label']), index


LOCKED_CONTRACT = {
    'revision_marker': REVISION_MARKER,
    'backbone': {'model': TIMM_MODEL_NAME, 'revision': V10_MODEL_REVISION,
                 'all_original_parameters_frozen': True, 'dtype': 'float16'},
    'data': {'official_raw_labels_only': True, 'dropped_rows': 0, 'relabeled_rows': 0,
             'fold': DEVELOPMENT_FOLD, 'fold_assignment_sha256': FOLD_SHA256},
    'preprocess': 'direct-square 256 bilinear antialias, ImageNet mean/std, no augmentation',
    'readout': 'final normalized CLS concatenated with mean of 256 true patch tokens',
    'head': 'exact V10 fold-0 head, rebuilt from verified cache then frozen',
    'only_change': {'target': 'block39.attn.fused_qkv.{query,value}', 'rank': LORA_RANK,
                    'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
                    'new_trainable_parameters': EXPECTED_ADAPTER_PARAMETERS},
    'training': {'epochs': LORA_EPOCHS, 'optimizer': 'AdamW', 'lr': LORA_LR,
                 'weight_decay': LORA_WEIGHT_DECAY, 'effective_batch': LORA_EFFECTIVE_BATCH,
                 'warmup_fraction': LORA_WARMUP_FRACTION, 'gradient_clip': GRAD_CLIP},
    'gate': {'min_macro_f1_gain': ADOPTION_MIN_GAIN, 'total_errors_must_decrease': True,
             'ro_errors_must_decrease': True,
             'max_electronic_f1_drop': ADOPTION_MAX_ELECTRONIC_F1_DROP},
    'test_access_during_selection': False,
}
LOCKED_CONTRACT_SHA256 = sha256_json(LOCKED_CONTRACT)
write_json_atomic({'locked_contract': LOCKED_CONTRACT, 'sha256': LOCKED_CONTRACT_SHA256},
                  ARTIFACT_DIR / 'locked_contract.json')

assert LORA_EFFECTIVE_BATCH == 16
assert EXPECTED_ADAPTER_PARAMETERS == 131072
assert len(TRAIN_INDICES) == 21221 and len(VALID_INDICES) == 5306
print('Contract locked:', LOCKED_CONTRACT_SHA256)
print('Official raw train:', TRAIN_DIR)
print('Fold 0:', len(TRAIN_INDICES), 'train /', len(VALID_INDICES), 'validation')
print('Test directories were pruned from discovery and were not enumerated.')

In [ ]:
class CachedRows(Dataset):
    def __init__(self, features, indices, labels=None):
        self.features = features
        self.indices = [int(i) for i in indices]
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        index = self.indices[position]
        label = int(self.labels[index]) if self.labels is not None else -1
        return self.features[index], label, index


class ChampionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.LayerNorm(FEATURE_DIM),
            nn.Linear(FEATURE_DIM, HEAD_HIDDEN),
            nn.GELU(),
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(HEAD_HIDDEN, NUM_CLASSES),
        )

    def forward(self, features):
        return self.network(features.flatten(1))


def softmax_numpy(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    probabilities = exponentials / exponentials.sum(axis=1, keepdims=True)
    if not np.isfinite(probabilities).all():
        raise RuntimeError('Probabilities contain NaN or Inf.')
    return probabilities


def metric_bundle(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    probabilities = np.asarray(probabilities, dtype=np.float64)
    if probabilities.shape != (len(labels), NUM_CLASSES) or not np.isfinite(probabilities).all():
        raise RuntimeError('Metric input is incomplete or non-finite.')
    predictions = probabilities.argmax(axis=1)
    matrix = confusion_matrix(labels, predictions, labels=list(range(NUM_CLASSES)))
    report = classification_report(labels, predictions, labels=list(range(NUM_CLASSES)),
                                   target_names=list(CLASS_NAMES), output_dict=True, zero_division=0)
    return {
        'rows': int(len(labels)),
        'macro_f1': float(f1_score(labels, predictions, labels=list(range(NUM_CLASSES)), average='macro')),
        'accuracy': float(accuracy_score(labels, predictions)),
        'log_loss': float(log_loss(labels, probabilities, labels=list(range(NUM_CLASSES)))),
        'f1_recyclable': float(report['Recyclable']['f1-score']),
        'f1_electronic': float(report['Electronic']['f1-score']),
        'f1_organic': float(report['Organic']['f1-score']),
        'confusion_matrix': matrix.tolist(),
        'total_errors': int((predictions != labels).sum()),
        'recyclable_to_organic': int(matrix[0, 2]),
        'organic_to_recyclable': int(matrix[2, 0]),
        'ro_errors': int(matrix[0, 2] + matrix[2, 0]),
        'electronic_errors': int(matrix[1].sum() - matrix[1, 1]),
    }


def predict_cached_head(model, features, indices, device='cuda:0'):
    index_list = [int(i) for i in indices]
    loader = DataLoader(CachedRows(features, index_list), batch_size=HEAD_PREDICT_BATCH,
                        shuffle=False, num_workers=0, pin_memory=True)
    logits_parts, observed = [], []
    model.eval()
    with torch.inference_mode():
        for batch_features, _labels, batch_indices in loader:
            logits = model(batch_features.to(device=device, dtype=torch.float32, non_blocking=True))
            logits_parts.append(logits.float().cpu())
            observed.extend(int(i) for i in batch_indices.tolist())
    if observed != index_list:
        raise RuntimeError('Cached prediction order changed.')
    return softmax_numpy(torch.cat(logits_parts).numpy().astype(np.float64))


def find_feature_tensor(payload, rows):
    candidates = [payload] if torch.is_tensor(payload) else (
        [value for value in payload.values() if torch.is_tensor(value)]
        if isinstance(payload, dict) else [])
    for tensor in candidates:
        if tensor.ndim >= 2 and int(tensor.shape[0]) == rows:
            flattened = int(np.prod(tuple(int(d) for d in tensor.shape[1:])))
            if flattened == FEATURE_DIM:
                return tensor.reshape(rows, 2, HIDDEN_SIZE)
    return None


def discover_train_cache_directories():
    candidates = []
    for directory in walk_directories_without_test(KAGGLE_INPUT):
        if directory.name.lower() == 'train' and any(directory.glob('train_000000_*.pt')):
            candidates.append(directory)
    return sorted({p.resolve() for p in candidates}, key=str)


def load_train_cache(directory):
    shards = sorted(directory.glob('train_*.pt'), key=lambda p: p.name)
    features = torch.empty((EXPECTED_ROWS, 2, HIDDEN_SIZE), dtype=FEATURE_DTYPE)
    covered = np.zeros(EXPECTED_ROWS, dtype=bool)
    for shard_path in shards:
        parts = shard_path.stem.split('_')
        try:
            start, end = int(parts[-2]), int(parts[-1])
        except (ValueError, IndexError):
            return None
        if start < 0 or end > EXPECTED_ROWS or end <= start:
            return None
        try:
            payload = torch.load(shard_path, map_location='cpu', weights_only=False)
        except Exception:
            return None
        tensor = find_feature_tensor(payload, end - start)
        if tensor is None:
            return None
        if isinstance(payload, dict) and isinstance(payload.get('image_ids'), list):
            if payload['image_ids'] != train_df.iloc[start:end]['image_id'].tolist():
                return None
        features[start:end] = tensor.to(FEATURE_DTYPE)
        covered[start:end] = True
        del payload, tensor
    if not covered.all() or not torch.isfinite(features.float()).all().item():
        return None
    return features


def train_v10_fold0_head(features):
    seed_everything(FOLD_SEED)
    device = 'cuda:0'
    model = ChampionHead().to(device)
    counts = np.bincount(LABEL_VALUES[TRAIN_INDICES], minlength=NUM_CLASSES)
    weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=HEAD_LR, weight_decay=HEAD_WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HEAD_MAX_EPOCHS)
    generator = torch.Generator().manual_seed(FOLD_SEED)
    loader = DataLoader(CachedRows(features, TRAIN_INDICES, LABEL_VALUES), batch_size=HEAD_BATCH,
                        shuffle=True, generator=generator, num_workers=0, pin_memory=True)
    best = {'macro_f1': -1.0, 'log_loss': float('inf'), 'epoch': None, 'state': None, 'stale': 0}
    for epoch in range(1, HEAD_MAX_EPOCHS + 1):
        model.train()
        for batch_features, batch_labels, _indices in loader:
            batch_features = batch_features.to(device=device, dtype=torch.float32, non_blocking=True)
            batch_labels = batch_labels.to(device=device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(batch_features), batch_labels)
            if not torch.isfinite(loss).item():
                raise RuntimeError('Non-finite V10 head loss.')
            loss.backward()
            gradient_norm = nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite V10 head gradient.')
            optimizer.step()
        scheduler.step()
        probabilities = predict_cached_head(model, features, VALID_INDICES, device=device)
        metrics = metric_bundle(LABEL_VALUES[VALID_INDICES], probabilities)
        improved = (metrics['macro_f1'] > best['macro_f1'] + 1e-8
                    or (abs(metrics['macro_f1'] - best['macro_f1']) <= 1e-8
                        and metrics['log_loss'] < best['log_loss']))
        if improved:
            best.update({'macro_f1': metrics['macro_f1'], 'log_loss': metrics['log_loss'],
                         'epoch': epoch, 'stale': 0,
                         'state': {name: tensor.detach().cpu().clone()
                                   for name, tensor in model.state_dict().items()}})
        else:
            best['stale'] += 1
        print('  V10 head epoch {:02d}: fold0 {:.10f}, error {}, R/O {}'.format(
            epoch, metrics['macro_f1'], metrics['total_errors'], metrics['ro_errors']))
        if best['stale'] >= HEAD_PATIENCE:
            break
    if best['state'] is None:
        raise RuntimeError('V10 head training produced no checkpoint.')
    model.load_state_dict(best['state'], strict=True)
    probabilities = predict_cached_head(model, features, VALID_INDICES, device=device)
    metrics = metric_bundle(LABEL_VALUES[VALID_INDICES], probabilities)
    metrics['best_epoch'] = best['epoch']
    model.to('cpu').eval().requires_grad_(False)
    del loader, criterion, optimizer, scheduler
    torch.cuda.empty_cache()
    return model, metrics


cache_candidates = discover_train_cache_directories()
print('Candidate train caches:', len(cache_candidates))
CONTROL_HEAD = None
CACHE_CONTROL_METRICS = None
CACHE_SOURCE = None
for candidate in cache_candidates:
    print('Trying cache:', candidate)
    cached_features = load_train_cache(candidate)
    if cached_features is None:
        print('  rejected: incomplete/incompatible shard schema')
        continue
    candidate_head, candidate_metrics = train_v10_fold0_head(cached_features)
    exact = (candidate_metrics['macro_f1'] == V10_FOLD0['macro_f1']
             and candidate_metrics['total_errors'] == V10_FOLD0['total_errors']
             and candidate_metrics['ro_errors'] == V10_FOLD0['ro_errors'])
    print('  control {:.10f}, error {}, R/O {}, exact={}'.format(
        candidate_metrics['macro_f1'], candidate_metrics['total_errors'],
        candidate_metrics['ro_errors'], exact))
    del cached_features
    gc.collect()
    if exact:
        CONTROL_HEAD = candidate_head
        CACHE_CONTROL_METRICS = candidate_metrics
        CACHE_SOURCE = str(candidate)
        break
    del candidate_head
if CONTROL_HEAD is None:
    raise RuntimeError('No attached train cache reproduced V10 fold 0 exactly. Attach timm-embedding-satdat.')
if sum(p.numel() for p in CONTROL_HEAD.parameters()) != 4212739:
    raise RuntimeError('Control head architecture is not the 4,212,739-parameter V10 head.')
write_json_atomic({'source': CACHE_SOURCE, 'metrics': CACHE_CONTROL_METRICS,
                   'exact_v10_fold0_reproduction': True}, ARTIFACT_DIR / 'cache_control.json')
atomic_torch_save({'state_dict': CONTROL_HEAD.state_dict(), 'metrics': CACHE_CONTROL_METRICS},
                  ARTIFACT_DIR / 'v10_fold0_control_head.pt')
print('Accepted cache:', CACHE_SOURCE)
print('Exact V10 fold-0 control head is frozen and ready.')

In [ ]:
import accelerate
import safetensors
import timm
from accelerate import dispatch_model, init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open

ENVIRONMENT = {
    'torch': torch.__version__, 'timm': timm.__version__,
    'accelerate': accelerate.__version__, 'safetensors': safetensors.__version__,
    'cuda': torch.version.cuda,
    'gpus': [{'device': device, 'name': torch.cuda.get_device_name(device),
              'vram_gib': torch.cuda.get_device_properties(device).total_memory / 2 ** 30}
             for device in (0, 1)],
}
write_json_atomic(ENVIRONMENT, ARTIFACT_DIR / 'environment.json')
print(json.dumps(ENVIRONMENT, indent=2))

SHARDING_STRATEGY = 'explicit_contiguous_two_gpu_streamed'
OUTPUT_SIDE_MODULES = ('norm', 'fc_norm', 'head_norm', 'attn_pool', 'head_drop', 'head', 'head_dist')
GPU_STATIC_LIMIT_GIB = 10.5
GPU_RUNTIME_HEADROOM_GIB = 3.0


def tensor_bytes(tensor):
    element_size = (torch.empty(0, dtype=BACKBONE_DTYPE).element_size()
                    if tensor.is_floating_point() else tensor.element_size())
    return tensor.numel() * element_size


def module_bytes(module):
    total = sum(tensor_bytes(parameter) for parameter in module.parameters(recurse=True))
    for submodule in module.modules():
        for name, buffer in submodule._buffers.items():
            if buffer is not None and name not in submodule._non_persistent_buffers_set:
                total += tensor_bytes(buffer)
    return total


def plan_two_gpu_placement(model):
    blocks = getattr(model, 'blocks', None)
    if blocks is None or len(blocks) != 40:
        raise RuntimeError('Expected exactly 40 ordered transformer blocks.')
    root_names, root_bytes = [], 0
    for name, parameter in model._parameters.items():
        if parameter is not None:
            root_names.append(name)
            root_bytes += tensor_bytes(parameter)
    for name, buffer in model._buffers.items():
        if buffer is not None and name not in model._non_persistent_buffers_set:
            root_names.append(name)
            root_bytes += tensor_bytes(buffer)
    block_sizes = [module_bytes(block) for block in blocks]
    input_side, output_side = [], []
    for child_name, child in model.named_children():
        if child_name == 'blocks':
            continue
        target = output_side if child_name in OUTPUT_SIDE_MODULES else input_side
        target.append((child_name, module_bytes(child)))
    input_bytes = root_bytes + sum(size for _, size in input_side)
    output_bytes = sum(size for _, size in output_side)
    candidates = []
    for split in range(1, len(blocks)):
        gpu0 = input_bytes + sum(block_sizes[:split])
        gpu1 = output_bytes + sum(block_sizes[split:])
        candidates.append((max(gpu0, gpu1), split, gpu0, gpu1))
    _, split_index, gpu0_bytes, gpu1_bytes = min(candidates)
    device_map = {name: 'cuda:0' for name in root_names}
    device_map.update({name: 'cuda:0' for name, _ in input_side})
    device_map.update({'blocks.{}'.format(index): ('cuda:0' if index < split_index else 'cuda:1')
                       for index in range(len(blocks))})
    device_map.update({name: 'cuda:1' for name, _ in output_side})
    return {'split_index': split_index, 'device_map': device_map,
            'static_gib': {'cuda:0': gpu0_bytes / 2 ** 30, 'cuda:1': gpu1_bytes / 2 ** 30}}


def enforce_placement_budget(plan):
    for device_id in (0, 1):
        key = 'cuda:{}'.format(device_id)
        planned = plan['static_gib'][key]
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_id)
        if (planned > GPU_STATIC_LIMIT_GIB
                or total_bytes / 2 ** 30 - planned < GPU_RUNTIME_HEADROOM_GIB
                or free_bytes / 2 ** 30 < planned):
            raise RuntimeError('Static placement leaves insufficient memory on {}: {}'.format(key, plan))


def target_device(tensor_name, device_map):
    matches = [(len(module_name), device) for module_name, device in device_map.items()
               if tensor_name == module_name or tensor_name.startswith(module_name + '.')]
    if not matches:
        raise RuntimeError('No device assignment for {}'.format(tensor_name))
    return max(matches)[1]


def resolve_hf_token():
    token = os.environ.get('HF_TOKEN')
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('Add a Kaggle Secret named HF_TOKEN with DINOv3 access.') from exc


def build_backbone():
    from huggingface_hub import model_info, snapshot_download

    token = resolve_hf_token()
    revision = model_info(TIMM_HF_REPO_ID, revision=V10_MODEL_REVISION, token=token).sha
    if revision != V10_MODEL_REVISION:
        raise RuntimeError('Hub revision differs from the immutable V10 revision.')
    snapshot_dir = snapshot_download(repo_id=TIMM_HF_REPO_ID, revision=V10_MODEL_REVISION,
                                     token=token,
                                     allow_patterns=['*.safetensors', '*.safetensors.index.json', '*.json'])
    del token
    with init_empty_weights(include_buffers=False):
        model = timm.create_model(TIMM_MODEL_NAME, pretrained=False, num_classes=0, img_size=IMAGE_SIZE)
    snapshot = Path(snapshot_dir)
    index_path = snapshot / 'model.safetensors.index.json'
    if index_path.is_file():
        checkpoint_names = sorted(set(read_json(index_path).get('weight_map', {}).values()))
    else:
        available = sorted(path.name for path in snapshot.glob('*.safetensors'))
        checkpoint_names = ['model.safetensors'] if 'model.safetensors' in available else available
    checkpoint_files = [snapshot / name for name in checkpoint_names]
    if not checkpoint_files or any(not path.is_file() for path in checkpoint_files):
        raise RuntimeError('DINOv3 safetensors checkpoint is incomplete.')

    plan = plan_two_gpu_placement(model)
    enforce_placement_budget(plan)
    print('Streaming model with split at block', plan['split_index'], plan['static_gib'])
    model_names = set(model.state_dict().keys())
    checkpoint_tensor_names = set()
    for checkpoint_file in checkpoint_files:
        with safe_open(str(checkpoint_file), framework='pt', device='cpu') as handle:
            keys = sorted(handle.keys())
            checkpoint_tensor_names.update(keys)
            for key in tqdm(keys, desc='streaming {}'.format(checkpoint_file.name)):
                if key not in model_names:
                    continue
                tensor = handle.get_tensor(key)
                if tensor.is_floating_point():
                    tensor = tensor.to(BACKBONE_DTYPE)
                set_module_tensor_to_device(model, key, target_device(key, plan['device_map']),
                                            value=tensor, dtype=BACKBONE_DTYPE)
                del tensor
    missing = sorted(model_names - checkpoint_tensor_names)
    if missing:
        raise RuntimeError('Checkpoint is missing model tensors: {}'.format(missing[:10]))
    meta_tensors = [name for name, tensor in list(model.named_parameters()) + list(model.named_buffers())
                    if tensor.device.type == 'meta']
    if meta_tensors:
        raise RuntimeError('Unmaterialized tensors remain: {}'.format(meta_tensors[:10]))

    model = dispatch_model(model, device_map=plan['device_map'], force_hooks=True)
    model.eval().requires_grad_(False)
    if sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad) != 0:
        raise RuntimeError('An original pretrained parameter is trainable.')
    if {parameter.dtype for parameter in model.parameters() if parameter.is_floating_point()} != {BACKBONE_DTYPE}:
        raise RuntimeError('Original floating parameters are not uniformly FP16.')
    if {str(parameter.device) for parameter in model.parameters()} != {'cuda:0', 'cuda:1'}:
        raise RuntimeError('Original parameters are not placed on exactly both GPUs.')
    if not [name for name, module in model.named_modules() if hasattr(module, '_hf_hook')]:
        raise RuntimeError('Accelerate dispatch hooks are missing.')
    if type(model).__name__ in {'DataParallel', 'DistributedDataParallel'}:
        raise RuntimeError('Replication wrapper detected; explicit sharding is required.')
    block_devices = []
    for block in model.blocks:
        devices = {str(parameter.device) for parameter in block.parameters()}
        if len(devices) != 1:
            raise RuntimeError('A transformer block was split across devices.')
        block_devices.append(devices.pop())
    transitions = [index for index in range(1, len(block_devices))
                   if block_devices[index] != block_devices[index - 1]]
    if (len(transitions) != 1 or block_devices[0] != 'cuda:0'
            or block_devices[-1] != 'cuda:1'):
        raise RuntimeError('Blocks are not two contiguous cuda:0/cuda:1 ranges.')
    if int(model.num_prefix_tokens) != PREFIX_TOKENS or int(model.num_features) != HIDDEN_SIZE:
        raise RuntimeError('Runtime token contract differs from V10.')
    return model, plan, block_devices


MODEL, PLACEMENT, BLOCK_DEVICES = build_backbone()
print('Backbone loaded:', BLOCK_DEVICES.count('cuda:0'), 'blocks on cuda:0 and',
      BLOCK_DEVICES.count('cuda:1'), 'on cuda:1')

In [ ]:
class FusedQVLoRA(nn.Module):
    def __init__(self, base, rank, alpha, dropout):
        super().__init__()
        if not isinstance(base, nn.Linear):
            raise TypeError('Expected timm fused qkv to be nn.Linear, got {}.'.format(type(base).__name__))
        if base.in_features != HIDDEN_SIZE or base.out_features != 3 * HIDDEN_SIZE or base.bias is not None:
            raise RuntimeError('Fused qkv shape/bias differs from the audited DINOv3 ViT-7B contract.')
        self.base = base
        self.rank = int(rank)
        self.alpha = int(alpha)
        self.scale = float(alpha) / float(rank)
        self.dropout = nn.Dropout(float(dropout))
        device = base.weight.device
        self.q_down = nn.Linear(HIDDEN_SIZE, rank, bias=False).to(device=device, dtype=torch.float32)
        self.q_up = nn.Linear(rank, HIDDEN_SIZE, bias=False).to(device=device, dtype=torch.float32)
        self.v_down = nn.Linear(HIDDEN_SIZE, rank, bias=False).to(device=device, dtype=torch.float32)
        self.v_up = nn.Linear(rank, HIDDEN_SIZE, bias=False).to(device=device, dtype=torch.float32)
        nn.init.kaiming_uniform_(self.q_down.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.v_down.weight, a=math.sqrt(5))
        nn.init.zeros_(self.q_up.weight)
        nn.init.zeros_(self.v_up.weight)
        self.base.requires_grad_(False)

    def forward(self, inputs):
        fused = self.base(inputs)
        query, key, value = fused.chunk(3, dim=-1)
        with torch.autocast(device_type='cuda', enabled=False):
            adapter_input = self.dropout(inputs.float())
            query_delta = self.q_up(self.q_down(adapter_input))
            value_delta = self.v_up(self.v_down(adapter_input))
        query = query + query_delta.to(query.dtype) * self.scale
        value = value + value_delta.to(value.dtype) * self.scale
        return torch.cat((query, key, value), dim=-1)


def adapter_named_parameters(adapter):
    return [(name, parameter) for name, parameter in adapter.named_parameters()
            if not name.startswith('base.')]


def adapter_state_cpu(adapter):
    return {name: parameter.detach().cpu().clone()
            for name, parameter in adapter_named_parameters(adapter)}


def load_adapter_state(adapter, state):
    named = dict(adapter_named_parameters(adapter))
    if set(named) != set(state):
        raise RuntimeError('Adapter checkpoint keys differ: {} vs {}.'.format(sorted(named), sorted(state)))
    with torch.no_grad():
        for name, parameter in named.items():
            parameter.copy_(state[name].to(device=parameter.device, dtype=parameter.dtype))


FINAL_ATTENTION = MODEL.blocks[LORA_BLOCK_INDEX].attn
if getattr(FINAL_ATTENTION, 'q_bias', None) is not None:
    raise RuntimeError('Expected q_bias=None for the audited DINOv3 ViT-7B attention.')
if getattr(FINAL_ATTENTION, 'v_bias', None) is not None:
    raise RuntimeError('Expected v_bias=None for the audited DINOv3 ViT-7B attention.')
seed_everything(FOLD_SEED)
ADAPTER = FusedQVLoRA(FINAL_ATTENTION.qkv, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
FINAL_ATTENTION.qkv = ADAPTER
ADAPTER_DEVICE = ADAPTER.base.weight.device
CONTROL_HEAD = CONTROL_HEAD.to(ADAPTER_DEVICE).eval().requires_grad_(False)

TRAINABLE = [parameter for _, parameter in adapter_named_parameters(ADAPTER)]
TRAINABLE_COUNT = sum(parameter.numel() for parameter in TRAINABLE)
ORIGINAL_TRAINABLE = sum(parameter.numel() for name, parameter in MODEL.named_parameters()
                         if parameter.requires_grad and '.qkv.base.' in name)
if TRAINABLE_COUNT != EXPECTED_ADAPTER_PARAMETERS or ORIGINAL_TRAINABLE != 0:
    raise RuntimeError('Freeze/parameter audit failed: adapter={}, original={}.'.format(
        TRAINABLE_COUNT, ORIGINAL_TRAINABLE))
all_trainable_names = [name for name, parameter in MODEL.named_parameters() if parameter.requires_grad]
if len(all_trainable_names) != 4 or not all('.attn.qkv.' in name for name in all_trainable_names):
    raise RuntimeError('Unexpected trainable tensors: {}'.format(all_trainable_names))
if sum(parameter.numel() for parameter in CONTROL_HEAD.parameters() if parameter.requires_grad) != 0:
    raise RuntimeError('The exact V10 control head must remain frozen during LoRA training.')
if torch.count_nonzero(ADAPTER.q_up.weight).item() or torch.count_nonzero(ADAPTER.v_up.weight).item():
    raise RuntimeError('LoRA up projections must start at exactly zero.')


def online_readout(pixel_values):
    input_device = next(MODEL.patch_embed.proj.parameters()).device
    model_inputs = pixel_values.to(input_device, dtype=BACKBONE_DTYPE, non_blocking=True)
    with torch.autocast(device_type='cuda', dtype=BACKBONE_DTYPE):
        tokens = MODEL.forward_features(model_inputs)
    if tuple(tokens.shape[1:]) != (SEQUENCE_TOKENS, HIDDEN_SIZE):
        raise RuntimeError('forward_features returned {}.'.format(tuple(tokens.shape)))
    cls = tokens[:, 0, :].float()
    patch_mean = tokens[:, PREFIX_TOKENS:, :].float().mean(dim=1)
    readout = torch.stack((cls, patch_mean), dim=1).to(FEATURE_DTYPE)
    readout = readout.to(ADAPTER_DEVICE).float()
    if not torch.isfinite(readout).all().item():
        raise RuntimeError('Online readout contains NaN or Inf.')
    return readout


def predict_online(indices):
    index_list = [int(i) for i in indices]
    loader = DataLoader(RawImageRows(train_df, index_list), batch_size=VALID_BATCH,
                        shuffle=False, num_workers=LOADER_WORKERS, pin_memory=True,
                        persistent_workers=False)
    logits_parts, observed = [], []
    MODEL.eval()
    CONTROL_HEAD.eval()
    with torch.inference_mode():
        for pixels, _labels, batch_indices in tqdm(loader, desc='fold0 validation'):
            readout = online_readout(pixels)
            with torch.autocast(device_type='cuda', enabled=False):
                logits = CONTROL_HEAD(readout.float())
            logits_parts.append(logits.float().cpu())
            observed.extend(int(i) for i in batch_indices.tolist())
    if observed != index_list:
        raise RuntimeError('Online validation order changed.')
    return softmax_numpy(torch.cat(logits_parts).numpy().astype(np.float64))


def lr_scale(step, total_steps, warmup_steps):
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps - 1))
    return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))


assert abs(lr_scale(0, 100, 10) - 0.1) < 1e-12
assert abs(lr_scale(10, 100, 10) - 1.0) < 1e-12
assert abs(lr_scale(99, 100, 10)) < 1e-12


def train_adapter_one_epoch():
    seed_everything(FOLD_SEED)
    MODEL.eval()
    ADAPTER.train()
    CONTROL_HEAD.eval()
    counts = np.bincount(LABEL_VALUES[TRAIN_INDICES], minlength=NUM_CLASSES)
    class_weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts),
                                 dtype=torch.float32, device=ADAPTER_DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(TRAINABLE, lr=LORA_LR, weight_decay=LORA_WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=True)
    generator = torch.Generator().manual_seed(FOLD_SEED)
    loader = DataLoader(RawImageRows(train_df, TRAIN_INDICES), batch_size=LORA_MICROBATCH,
                        shuffle=True, generator=generator, num_workers=LOADER_WORKERS,
                        pin_memory=True, persistent_workers=False, drop_last=False)
    total_steps = math.ceil(len(loader) / LORA_GRAD_ACCUM)
    warmup_steps = max(1, round(total_steps * LORA_WARMUP_FRACTION))
    optimizer.zero_grad(set_to_none=True)
    optimizer_step = 0
    loss_sum = 0.0
    rows_seen = 0
    started = time.perf_counter()
    for micro_step, (pixels, labels, _indices) in enumerate(tqdm(loader, desc='LoRA epoch 1/1')):
        if micro_step % LORA_GRAD_ACCUM == 0:
            group_rows = min(LORA_EFFECTIVE_BATCH, len(TRAIN_INDICES) - rows_seen)
            scale = lr_scale(optimizer_step, total_steps, warmup_steps)
            for group in optimizer.param_groups:
                group['lr'] = LORA_LR * scale
        labels = labels.to(ADAPTER_DEVICE, non_blocking=True)
        readout = online_readout(pixels)
        with torch.autocast(device_type='cuda', enabled=False):
            logits = CONTROL_HEAD(readout.float())
            loss = criterion(logits, labels)
        if not torch.isfinite(loss).item():
            raise RuntimeError('Non-finite LoRA loss at micro-step {}.'.format(micro_step))
        scaler.scale(loss * len(labels) / group_rows).backward()
        loss_sum += loss.item() * len(labels)
        rows_seen += len(labels)
        end_of_group = ((micro_step + 1) % LORA_GRAD_ACCUM == 0 or micro_step + 1 == len(loader))
        if end_of_group:
            scaler.unscale_(optimizer)
            gradient_norm = nn.utils.clip_grad_norm_(TRAINABLE, GRAD_CLIP)
            if not torch.isfinite(gradient_norm).item():
                raise RuntimeError('Non-finite LoRA gradient at optimizer step {}.'.format(optimizer_step))
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            optimizer_step += 1
            if optimizer_step % 100 == 0 or optimizer_step == total_steps:
                print('  step {:04d}/{:04d} lr {:.2e} mean_loss {:.6f}'.format(
                    optimizer_step, total_steps, optimizer.param_groups[0]['lr'], loss_sum / rows_seen))
    if rows_seen != len(TRAIN_INDICES) or optimizer_step != total_steps:
        raise RuntimeError('Training coverage/step count mismatch.')
    ADAPTER.eval()
    return {'epochs': 1, 'rows_seen': rows_seen, 'optimizer_steps': optimizer_step,
            'warmup_steps': warmup_steps, 'mean_loss': loss_sum / rows_seen,
            'training_seconds': time.perf_counter() - started,
            'peak_cuda_gib': {str(device): torch.cuda.max_memory_allocated(device) / 2 ** 30
                              for device in (0, 1)}}


print('LoRA attached to block {} Q/V: {:,} trainable parameters on {}.'.format(
    LORA_BLOCK_INDEX, TRAINABLE_COUNT, ADAPTER_DEVICE))
print('All original pretrained parameters and all head parameters are frozen.')

In [ ]:
print('=' * 78)
print('LIVE ZERO-LoRA CONTROL — must reproduce V10 fold 0')
print('=' * 78)
CONTROL_LIVE_PROBABILITIES = predict_online(VALID_INDICES)
CONTROL_LIVE_METRICS = metric_bundle(LABEL_VALUES[VALID_INDICES], CONTROL_LIVE_PROBABILITIES)
CONTROL_INTEGRITY = (
    CONTROL_LIVE_METRICS['macro_f1'] == V10_FOLD0['macro_f1']
    and CONTROL_LIVE_METRICS['total_errors'] == V10_FOLD0['total_errors']
    and CONTROL_LIVE_METRICS['ro_errors'] == V10_FOLD0['ro_errors']
    and CONTROL_LIVE_METRICS['f1_electronic'] == V10_FOLD0['f1_electronic']
)
print('control macro_f1={:.10f} errors={} R/O={} ElectronicF1={:.10f} integrity={}'.format(
    CONTROL_LIVE_METRICS['macro_f1'], CONTROL_LIVE_METRICS['total_errors'],
    CONTROL_LIVE_METRICS['ro_errors'], CONTROL_LIVE_METRICS['f1_electronic'], CONTROL_INTEGRITY))
if not CONTROL_INTEGRITY:
    raise RuntimeError('Live zero-LoRA backbone/head did not reproduce V10 fold 0 exactly.')
write_json_atomic(CONTROL_LIVE_METRICS, ARTIFACT_DIR / 'fold0_control_live_metrics.json')

print()
print('=' * 78)
print('ONE RAW-LABEL EPOCH — final-block Q/V LoRA only')
print('=' * 78)
TRAINING_HISTORY = train_adapter_one_epoch()
write_json_atomic(TRAINING_HISTORY, ARTIFACT_DIR / 'training_history.json')

print()
print('=' * 78)
print('CANDIDATE VALIDATION — one deterministic pass')
print('=' * 78)
CANDIDATE_PROBABILITIES = predict_online(VALID_INDICES)
CANDIDATE_METRICS = metric_bundle(LABEL_VALUES[VALID_INDICES], CANDIDATE_PROBABILITIES)
GAIN = CANDIDATE_METRICS['macro_f1'] - CONTROL_LIVE_METRICS['macro_f1']
ELECTRONIC_F1_DROP = CONTROL_LIVE_METRICS['f1_electronic'] - CANDIDATE_METRICS['f1_electronic']
control_predictions = CONTROL_LIVE_PROBABILITIES.argmax(axis=1)
candidate_predictions = CANDIDATE_PROBABILITIES.argmax(axis=1)
valid_labels = LABEL_VALUES[VALID_INDICES]
CONTROL_CORRECT = control_predictions == valid_labels
CANDIDATE_CORRECT = candidate_predictions == valid_labels
FIXED_ROWS = int((~CONTROL_CORRECT & CANDIDATE_CORRECT).sum())
BROKEN_ROWS = int((CONTROL_CORRECT & ~CANDIDATE_CORRECT).sum())

GATE_CHECKS = {
    'live_control_integrity': CONTROL_INTEGRITY,
    'macro_f1_gain_at_least_0.0010': bool(GAIN >= ADOPTION_MIN_GAIN),
    'total_errors_decreased': bool(CANDIDATE_METRICS['total_errors'] < CONTROL_LIVE_METRICS['total_errors']),
    'ro_errors_decreased': bool(CANDIDATE_METRICS['ro_errors'] < CONTROL_LIVE_METRICS['ro_errors']),
    'electronic_f1_drop_at_most_0.0005': bool(
        ELECTRONIC_F1_DROP <= ADOPTION_MAX_ELECTRONIC_F1_DROP),
    'test_untouched': bool(not any(TEST_ACCESS_STATE.values())),
}
CORE_GATE_PASS = bool(all(GATE_CHECKS.values()))
DECISION = 'PENDING_CHECKPOINT_RELOAD'

adapter_checkpoint_state = adapter_state_cpu(ADAPTER)
if any(name.startswith('base.') for name in adapter_checkpoint_state):
    raise RuntimeError('Checkpoint unexpectedly contains original qkv weights.')
CHECKPOINT_PATH = ARTIFACT_DIR / 'fold0_adapter_and_frozen_head.pt'
atomic_torch_save({
    'metadata': {'revision_marker': REVISION_MARKER, 'locked_contract_sha256': LOCKED_CONTRACT_SHA256,
                 'model_revision': V10_MODEL_REVISION, 'block_index': LORA_BLOCK_INDEX,
                 'rank': LORA_RANK, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT,
                 'decision': DECISION},
    'adapter_state': adapter_checkpoint_state,
    'frozen_head_state': {name: tensor.detach().cpu().clone()
                          for name, tensor in CONTROL_HEAD.state_dict().items()},
    'control_metrics': CONTROL_LIVE_METRICS,
    'candidate_metrics': CANDIDATE_METRICS,
}, CHECKPOINT_PATH)

reloaded_payload = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
base_qkv = ADAPTER.base
MODEL.blocks[LORA_BLOCK_INDEX].attn.qkv = base_qkv
del ADAPTER
gc.collect()
RELOADED_ADAPTER = FusedQVLoRA(base_qkv, LORA_RANK, LORA_ALPHA, LORA_DROPOUT)
load_adapter_state(RELOADED_ADAPTER, reloaded_payload['adapter_state'])
MODEL.blocks[LORA_BLOCK_INDEX].attn.qkv = RELOADED_ADAPTER
ADAPTER = RELOADED_ADAPTER
ADAPTER.eval()
RELOADED_HEAD = ChampionHead().to(ADAPTER_DEVICE).eval().requires_grad_(False)
RELOADED_HEAD.load_state_dict(reloaded_payload['frozen_head_state'], strict=True)
CONTROL_HEAD = RELOADED_HEAD
RELOADED_PROBE = predict_online(VALID_INDICES[:VALID_BATCH])
PROBE_MAX_ABS_DIFFERENCE = float(np.abs(
    RELOADED_PROBE - CANDIDATE_PROBABILITIES[:VALID_BATCH]).max())
PROBE_ARGMAX_IDENTICAL = bool(np.array_equal(
    RELOADED_PROBE.argmax(axis=1), CANDIDATE_PROBABILITIES[:VALID_BATCH].argmax(axis=1)))
REPRODUCTION_PASS = bool(PROBE_MAX_ABS_DIFFERENCE <= RELOAD_TOLERANCE and PROBE_ARGMAX_IDENTICAL)
if not REPRODUCTION_PASS:
    raise RuntimeError('Reloaded adapter failed the held-out-batch reproduction check.')
GATE_CHECKS['checkpoint_reload_reproduction'] = REPRODUCTION_PASS
FOLD0_PASS = bool(CORE_GATE_PASS and all(GATE_CHECKS.values()))
DECISION = 'PASS_FOLD0_SCREEN_WAIT_FOR_CONFIRMATION' if FOLD0_PASS else 'REJECT_KEEP_V10_CHAMPION'
reloaded_payload['metadata']['decision'] = DECISION
reloaded_payload['metadata']['checkpoint_reproduction_pass'] = REPRODUCTION_PASS
atomic_torch_save(reloaded_payload, CHECKPOINT_PATH)

write_npy_atomic(CONTROL_LIVE_PROBABILITIES.astype(np.float32),
                 ARTIFACT_DIR / 'fold0_control_probabilities.npy')
write_npy_atomic(CANDIDATE_PROBABILITIES.astype(np.float32),
                 ARTIFACT_DIR / 'fold0_candidate_probabilities.npy')
prediction_frame = train_df.iloc[VALID_INDICES][['image_id', 'label']].copy()
prediction_frame['control_prediction'] = control_predictions
prediction_frame['candidate_prediction'] = candidate_predictions
prediction_frame['fixed_by_lora'] = (~CONTROL_CORRECT & CANDIDATE_CORRECT)
prediction_frame['broken_by_lora'] = (CONTROL_CORRECT & ~CANDIDATE_CORRECT)
write_csv_atomic(prediction_frame, ARTIFACT_DIR / 'fold0_predictions.csv')

SELECTION_DECISION = {
    'decision': DECISION,
    'fold0_pass': FOLD0_PASS,
    'gate_checks': GATE_CHECKS,
    'gain': GAIN,
    'electronic_f1_drop': ELECTRONIC_F1_DROP,
    'fixed_rows': FIXED_ROWS,
    'broken_rows': BROKEN_ROWS,
    'control_metrics': CONTROL_LIVE_METRICS,
    'candidate_metrics': CANDIDATE_METRICS,
    'training_history': TRAINING_HISTORY,
    'control_cache_source': CACHE_SOURCE,
    'checkpoint_reproduction': {'rows': VALID_BATCH,
                                'max_abs_probability_difference': PROBE_MAX_ABS_DIFFERENCE,
                                'argmax_identical': PROBE_ARGMAX_IDENTICAL,
                                'pass': REPRODUCTION_PASS},
    'original_pretrained_parameters_trainable': 0,
    'adapter_parameters_trainable': TRAINABLE_COUNT,
    'head_parameters_trainable': 0,
    'test_images_accessed': False,
    'full_oof_claimed': False,
    'submitted_to_organisers': False,
}
write_json_atomic(SELECTION_DECISION, ARTIFACT_DIR / 'selection_decision.json')
write_json_atomic({
    'backbone': 'DINOv3 ViT-7B/16 timm LVD-1689M',
    'repository': TIMM_HF_REPO_ID,
    'immutable_revision': V10_MODEL_REVISION,
    'original_pretrained_weights_frozen': True,
    'new_adapter': 'rank-8 alpha-16 Q/V LoRA in transformer block 39 only',
    'placement': PLACEMENT,
    'block_devices': BLOCK_DEVICES,
    'automatic_device_map_used': False,
    'cpu_or_disk_offload_used': False,
    'data_parallel_used': False,
    'environment': ENVIRONMENT,
}, ARTIFACT_DIR / 'model_manifest.json')

print()
print('#' * 78)
print('# V16 RESULT BLOCK — copy this whole block back')
print('#' * 78)
print('experiment              :', EXPERIMENT_NAME)
print('live control integrity  :', 'PASS' if CONTROL_INTEGRITY else 'FAIL')
print('control fold0           : macro_f1={:.10f} errors={} R/O={} ElectronicF1={:.10f}'.format(
    CONTROL_LIVE_METRICS['macro_f1'], CONTROL_LIVE_METRICS['total_errors'],
    CONTROL_LIVE_METRICS['ro_errors'], CONTROL_LIVE_METRICS['f1_electronic']))
print('candidate fold0         : macro_f1={:.10f} errors={} R/O={} ElectronicF1={:.10f}'.format(
    CANDIDATE_METRICS['macro_f1'], CANDIDATE_METRICS['total_errors'],
    CANDIDATE_METRICS['ro_errors'], CANDIDATE_METRICS['f1_electronic']))
print('gain                    : {:+.10f}'.format(GAIN))
print('fixed / broken rows     : {} / {}'.format(FIXED_ROWS, BROKEN_ROWS))
print('Electronic F1 drop      : {:+.10f}'.format(ELECTRONIC_F1_DROP))
print('train seconds           : {:.1f}'.format(TRAINING_HISTORY['training_seconds']))
print('trainable adapter params:', TRAINABLE_COUNT)
print('reload max abs diff     : {:.3e} (argmax identical: {})'.format(
    PROBE_MAX_ABS_DIFFERENCE, PROBE_ARGMAX_IDENTICAL))
for name, passed in GATE_CHECKS.items():
    print('gate {:38s}: {}'.format(name, 'PASS' if passed else 'FAIL'))
print('DECISION                :', DECISION)
print('test accessed           : False')
print('full OOF claimed        : False')
print('artifact directory      :', ARTIFACT_DIR)
print('elapsed minutes         : {:.1f}'.format((time.perf_counter() - NOTEBOOK_STARTED) / 60.0))
print('#' * 78)

## Manual handoff gate — berhenti di sini

Notebook sudah menulis keputusan otomatis, tetapi user tetap harus membaca dan mengirim balik seluruh `V16 RESULT BLOCK`.

- **PASS fold 0** hanya bila gain ≥ `+0.0010`, total error turun, error R/O turun, Electronic F1 tidak turun lebih dari `0.0005`, kontrol live exact, reload checkpoint lulus, dan test tetap untouched.
- Jika PASS: **jangan inference test dan jangan submit**. Langkah berikutnya baru confirmation folds 1–4; adopsi full membutuhkan gain OOF ≥ `+0.0003`, menang ≥3/4 confirmation fold, dan stabil lintas seed.
- Jika FAIL: V10 tetap champion. Jangan melonggarkan gate atau menambah epoch secara post-hoc; lanjut ke eksperimen V12B aspect-preserving hanya setelah hasil ini dilaporkan.

Target >0.997 tetap sangat tidak realistis tanpa breakthrough besar dari LoRA atau V12B. Screen ini tidak mengklaim full OOF dan tidak membuat submission.